In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Carga del conjunto de datos
df = pd.read_csv('kc_house_data.csv')

# Inspección inicial de la estructura
print("--- RESUMEN DEL CONJUNTO DE DATOS ---")
print(f"Total de observaciones: {df.shape[0]}")
print(f"Total de variables: {df.shape[1]}")
print(f"Valores nulos en el dataset: {df.isnull().sum().sum()}")
print("\nPrimeros registros:")
display(df.head(3))

--- RESUMEN DEL CONJUNTO DE DATOS ---
Total de observaciones: 21613
Total de variables: 21
Valores nulos en el dataset: 0

Primeros registros:


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062


In [2]:
# 1. Selección de variables independientes
cols_X = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'condition', 'grade', 'yr_built']

# 2. Construcción del Vector Y (Precio)
y = df['price'].values.reshape(-1, 1)

# 3. Construcción de la Matriz X
X_raw = df[cols_X].values

# 4. Adición de la columna de unos para el cálculo del Intercepto
unos = np.ones((X_raw.shape[0], 1))
X = np.append(unos, X_raw, axis=1)

print("--- DIMENSIONES DE LAS MATRICES ---")
print(f"Matriz X (con columna de unos): {X.shape}")
print(f"Vector Y: {y.shape}")
print("\nPrimer registro de la Matriz X (debe iniciar con 1.0):")
print(X[0])

--- DIMENSIONES DE LAS MATRICES ---
Matriz X (con columna de unos): (21613, 9)
Vector Y: (21613, 1)

Primer registro de la Matriz X (debe iniciar con 1.0):
[1.000e+00 3.000e+00 1.000e+00 1.180e+03 5.650e+03 1.000e+00 3.000e+00
 7.000e+00 1.955e+03]


In [3]:
# 1. Transpuesta de X
X_T = X.T

# 2. Multiplicación de X transpuesta por X
X_T_X = np.dot(X_T, X)

# 3. Cálculo de la matriz inversa
X_T_X_inv = np.linalg.inv(X_T_X)

# 4. Resolución del vector de coeficientes (b)
b = np.dot(np.dot(X_T_X_inv, X_T), y)

# Presentación de los resultados
nombres_coeficientes = ['Intercepto (Beta 0)'] + cols_X

print("--- VECTOR DE COEFICIENTES (b) CALCULADOS MANUALMENTE ---")
for nombre, coef in zip(nombres_coeficientes, b):
    # Formateamos a 4 decimales para mayor legibilidad
    print(f"{nombre}: {coef[0]:,.4f}")

--- VECTOR DE COEFICIENTES (b) CALCULADOS MANUALMENTE ---
Intercepto (Beta 0): 6,998,537.2939
bedrooms: -49,278.6922
bathrooms: 52,633.1348
sqft_living: 188.0648
sqft_lot: -0.2413
floors: 21,458.4360
condition: 19,183.0239
grade: 130,033.0441
yr_built: -4,000.1267


In [4]:
# 1. Calcular las predicciones del modelo (Y_estimada)
y_pred = np.dot(X, b)

# 2. Calcular los residuales (Errores = Real - Predicción)
residuales = y - y_pred

# 3. Cálculo manual de las Sumas de Cuadrados
RSS = np.sum(residuales**2)
TSS = np.sum((y - np.mean(y))**2)
ESS = TSS - RSS

# 4. Cálculo del R Cuadrado
R2 = 1 - (RSS / TSS)

# 5. Cálculo del R Cuadrado Ajustado
n = X.shape[0]      # Total de filas (21613)
k = X.shape[1] - 1  # Total de variables independientes sin el intercepto (8)
R2_adj = 1 - ((RSS / (n - k - 1)) / (TSS / (n - 1)))

# Presentación de Métricas
print("--- MÉTRICAS DE BONDAD DE AJUSTE (CÁLCULO MANUAL) ---")
print(f"Suma Total de Cuadrados (TSS): {TSS:,.2f}")
print(f"Suma de Cuadrados Residuales (RSS): {RSS:,.2f}")
print(f"Coeficiente de Determinación (R^2): {R2:.4f}")
print(f"R^2 Ajustado: {R2_adj:.4f}")

--- MÉTRICAS DE BONDAD DE AJUSTE (CÁLCULO MANUAL) ---
Suma Total de Cuadrados (TSS): 2,912,916,761,921,299.00
Suma de Cuadrados Residuales (RSS): 1,114,849,623,441,132.75
Coeficiente de Determinación (R^2): 0.6173
R^2 Ajustado: 0.6171


In [5]:
# 1. Preparación de la matriz X con la constante usando statsmodels
X_df_sm = sm.add_constant(df[cols_X])

# 2. Ajuste del modelo mediante OLS (Mínimos Cuadrados Ordinarios)
modelo_automatizado = sm.OLS(y, X_df_sm).fit()

# 3. Impresión del reporte automatizado
print(modelo_automatizado.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.617
Model:                            OLS   Adj. R-squared:                  0.617
Method:                 Least Squares   F-statistic:                     4355.
Date:                Mon, 13 Apr 2026   Prob (F-statistic):               0.00
Time:                        18:13:59   Log-Likelihood:            -2.9723e+05
No. Observations:               21613   AIC:                         5.945e+05
Df Residuals:                   21604   BIC:                         5.945e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const        6.999e+06   1.34e+05     52.046      